In [49]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler 


Up untill now I am just downloading the dataset, but in this dataset the index is multi-index so wi first flattened it

In [50]:
# now because csv dont parse the date time so this might be an issue for me, as the data will be trained in the future project based on 
# rolling window approach so it is really crucial to make sure that the data has the datetime sorted 

FinalStocks = pd.read_csv('data/raw_stocks.csv', parse_dates=["Date"])
# this is the first step to clean the data cleaning and making sure that the date column is of type datetime only 

In [51]:
FinalStocks[FinalStocks["Ticker"] == 'CBA.AX'].head()

,Date,Close,High,Low,Open,Volume,Ticker
4109,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX
4110,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX
4111,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX
4112,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX
4113,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX


In [52]:
print(FinalStocks['Ticker'].unique())

['BHP.AX' 'CBA.AX' 'CSL.AX' 'NAB.AX' 'WBC.AX' 'ANZ.AX' 'WES.AX' 'MQG.AX'
 'WOW.AX' 'TLS.AX']


In [53]:
CBA_Stock = FinalStocks[FinalStocks['Ticker'] == 'CBA.AX'].copy()

In [54]:
CBA_Stock = CBA_Stock.sort_values('Date').reset_index(drop=True)

In [55]:
print("Shape:", CBA_Stock.shape)

print("\nDate Range:")
print(CBA_Stock['Date'].min(), "→", CBA_Stock['Date'].max())

print("\nCheck ordering:")
print(CBA_Stock['Date'].is_monotonic_increasing)

print("\nMissing values:")
print(CBA_Stock.isnull().sum())

Shape: (4109, 7)

Date Range:
2010-01-04 00:00:00 → 2026-04-02 00:00:00

Check ordering:
True

Missing values:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Ticker    0
dtype: int64


In [56]:
CBA_Stock['Return'] = CBA_Stock['Close'].pct_change()

CBA_Stock['LogReturn'] = np.log(CBA_Stock['Close'] / CBA_Stock['Close'].shift(1))

In [57]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904


In [58]:
CBA_Stock['SMA_5'] = CBA_Stock['Close'].rolling(5).mean()
CBA_Stock['SMA_10'] = CBA_Stock['Close'].rolling(10).mean()

In [59]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721


In [60]:
CBA_Stock['Return_std_5'] = CBA_Stock['Return'].rolling(5).std()
CBA_Stock['Return_std_10'] = CBA_Stock['Return'].rolling(10).std()

In [61]:
CBA_Stock['Return_lag1'] = CBA_Stock['Return'].shift(1)
CBA_Stock['Return_lag2'] = CBA_Stock['Return'].shift(2)
CBA_Stock['Return_lag3'] = CBA_Stock['Return'].shift(3)
CBA_Stock['Return_lag5'] = CBA_Stock['Return'].shift(5)

In [62]:
CBA_Stock['Momentum_5'] = CBA_Stock['Return'].rolling(5).sum()
CBA_Stock['Momentum_10'] = CBA_Stock['Return'].rolling(10).sum()

In [63]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10,Return_std_5,Return_std_10,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN,NaN,NaN,0.015126,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN,NaN,NaN,0.005027,0.015126,NaN,NaN,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN,NaN,NaN,-0.009646,0.005027,0.015126,NaN,NaN,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN,0.009739,NaN,0.012987,-0.009646,0.005027,NaN,0.030795,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN,0.008532,NaN,0.007300,0.012987,-0.009646,0.015126,0.015668,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN,0.010160,NaN,0.000000,0.007300,0.012987,0.005027,0.000742,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN,0.009946,NaN,-0.009899,0.000000,0.007300,-0.009646,0.024314,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721,0.012656,NaN,0.013926,-0.009899,0.000000,0.012987,0.034395,NaN


In [64]:
CBA_Stock['Close_SMA5_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_5']
CBA_Stock['Close_SMA10_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_10']

In [65]:
CBA_Stock['TargetReturn'] = CBA_Stock['Return'].shift(-1)
CBA_Stock['TargetTrend'] = (CBA_Stock['TargetReturn'] > 0).astype(int)

In [66]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015126,1
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005027,1
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,...,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009646,0
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,...,0.005027,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,0.012987,1
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,...,-0.009646,0.005027,0.015126,NaN,NaN,NaN,0.237885,NaN,0.007300,1


In [67]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

In [68]:
print(CBA_Stock['TargetTrend'].value_counts(normalize=True))

TargetTrend
1    0.526354
0    0.473646
Name: proportion, dtype: float64


In [69]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,0.023067,0.013926,-0.009899,0.007300,0.026234,0.057028,0.424591,0.679077,-0.024289,0
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,-0.000861,0.023067,0.013926,0.000000,0.001944,0.017613,-0.214539,0.004040,-0.001412,0
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,-0.024289,-0.000861,0.023067,-0.009899,0.010432,0.011174,-0.299812,-0.057893,0.000000,0
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,-0.001412,-0.024289,-0.000861,0.013926,-0.003495,0.020820,-0.279167,-0.108163,-0.014321,0
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,0.000000,-0.001412,-0.024289,0.023067,-0.040883,-0.006489,-0.431776,-0.453318,-0.010762,0
5,2010-01-25,24.752924,24.842689,24.577881,24.685599,3028846,CBA.AX,-0.010762,-0.010821,25.193676,...,-0.014321,0.000000,-0.001412,-0.000861,-0.050785,-0.024552,-0.440752,-0.658881,-0.009429,0
6,2010-01-27,24.519535,24.645208,24.326538,24.604813,4035586,CBA.AX,-0.009429,-0.009473,25.013247,...,-0.010762,-0.014321,0.000000,-0.024289,-0.035925,-0.033980,-0.493712,-0.805197,0.008603,1
7,2010-01-28,24.730480,24.775363,24.550948,24.721503,5277686,CBA.AX,0.008603,0.008566,24.882188,...,-0.009429,-0.010762,-0.014321,-0.001412,-0.025909,-0.015478,-0.151707,-0.553409,-0.033938,0
8,2010-01-29,23.891169,24.434252,23.787938,24.371415,9940798,CBA.AX,-0.033938,-0.034528,24.583266,...,0.008603,-0.009429,-0.010762,0.000000,-0.059848,-0.063342,-0.692097,-1.232937,-0.004697,0
9,2010-02-01,23.778961,24.120072,23.778961,23.877704,4273994,CBA.AX,-0.004697,-0.004708,24.334614,...,-0.033938,0.008603,-0.009429,-0.014321,-0.050223,-0.091106,-0.555653,-1.115344,0.010759,1


In [70]:
CBA_Stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              4098 non-null   datetime64[ns]
 1   Close             4098 non-null   float64       
 2   High              4098 non-null   float64       
 3   Low               4098 non-null   float64       
 4   Open              4098 non-null   float64       
 5   Volume            4098 non-null   int64         
 6   Ticker            4098 non-null   object        
 7   Return            4098 non-null   float64       
 8   LogReturn         4098 non-null   float64       
 9   SMA_5             4098 non-null   float64       
 10  SMA_10            4098 non-null   float64       
 11  Return_std_5      4098 non-null   float64       
 12  Return_std_10     4098 non-null   float64       
 13  Return_lag1       4098 non-null   float64       
 14  Return_lag2       4098 n

In [71]:
feature_columns = [
    'Return',
    'Volume',

    'Return_lag1',
    'Return_lag2',
    'Return_lag3',
    'Return_lag5',

    'Momentum_5',
    'Momentum_10',

    'Return_std_5',
    'Return_std_10',

    'Close_SMA5_diff',
    'Close_SMA10_diff'
]

In [72]:
TrainingEnd = '2021-12-31'
ValidationEnd = '2023-12-31'

Train = CBA_Stock[CBA_Stock['Date'] <= TrainingEnd]
Validation = CBA_Stock[(CBA_Stock['Date'] > TrainingEnd) & (CBA_Stock['Date'] <= ValidationEnd)]
Test = CBA_Stock[CBA_Stock['Date'] > ValidationEnd]

In [73]:
X_train = Train[feature_columns]
X_val = Validation[feature_columns]
X_test = Test[feature_columns]

y_train = Train['TargetTrend']
y_train_reg = Train['TargetReturn']
y_val = Validation['TargetTrend']
y_test = Test['TargetTrend']

In [74]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [75]:
from sklearn.linear_model import LogisticRegression

In [76]:
LRModel = LogisticRegression(max_iter=1000, random_state=42)
LRModel.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [77]:
y_val_pred = LRModel.predict(X_val_scaled)
y_val_prob = LRModel.predict_proba(X_val_scaled)[:, 1]

y_test_pred = LRModel.predict(X_test_scaled)
y_test_prob = LRModel.predict_proba(X_test_scaled)[:, 1]

In [78]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

def ClassificationEvaluation(model_name, y_true, y_pred, y_prob):
    print(model_name)
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")
    print(f"AUC      : {roc_auc_score(y_true, y_prob):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [79]:
ClassificationEvaluation(
    "Logistic Regression - Validation",
    y_val,
    y_val_pred,
    y_val_prob
)
ClassificationEvaluation(
    "Logistic Regression - Test",
    y_test,
    y_test_pred,
    y_test_prob
)

Logistic Regression - Validation
Accuracy : 0.5129
F1 Score : 0.6293
AUC      : 0.5065

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.21      0.29       237
           1       0.53      0.78      0.63       266

    accuracy                           0.51       503
   macro avg       0.49      0.50      0.46       503
weighted avg       0.50      0.51      0.47       503

Confusion Matrix:
[[ 50 187]
 [ 58 208]]
Logistic Regression - Test
Accuracy : 0.5035
F1 Score : 0.6075
AUC      : 0.4551

Classification Report:
              precision    recall  f1-score   support

           0       0.40      0.28      0.32       247
           1       0.55      0.68      0.61       323

    accuracy                           0.50       570
   macro avg       0.47      0.48      0.47       570
weighted avg       0.48      0.50      0.48       570

Confusion Matrix:
[[ 68 179]
 [104 219]]


In [80]:
coef_df = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': LRModel.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coef_df)

             Feature  Coefficient
11  Close_SMA10_diff     0.293344
2        Return_lag1     0.136874
0             Return     0.125660
3        Return_lag2     0.049394
4        Return_lag3     0.009804
1             Volume     0.005517
8       Return_std_5     0.003589
5        Return_lag5    -0.022388
9      Return_std_10    -0.041203
7        Momentum_10    -0.050345
6         Momentum_5    -0.197610
10   Close_SMA5_diff    -0.259841


In [81]:
from xgboost import XGBClassifier

XBGclass = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

XBGclass.fit(X_train, y_train)

y_val_pred = XBGclass.predict(X_val)
y_val_prob = XBGclass.predict_proba(X_val)[:, 1]

y_test_pred = XBGclass.predict(X_test)
y_test_prob = XBGclass.predict_proba(X_test)[:, 1]

In [82]:
ClassificationEvaluation(
    "XGB - Validation",
    y_val,
    y_val_pred,
    y_val_prob
)
ClassificationEvaluation(
    "XGB - Test",
    y_test,
    y_test_pred,
    y_test_prob
)

XGB - Validation
Accuracy : 0.5129
F1 Score : 0.5694
AUC      : 0.4909

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.41      0.44       237
           1       0.53      0.61      0.57       266

    accuracy                           0.51       503
   macro avg       0.51      0.51      0.50       503
weighted avg       0.51      0.51      0.51       503

Confusion Matrix:
[[ 96 141]
 [104 162]]
XGB - Test
Accuracy : 0.5228
F1 Score : 0.5976
AUC      : 0.5221

Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.39      0.41       247
           1       0.57      0.63      0.60       323

    accuracy                           0.52       570
   macro avg       0.51      0.51      0.51       570
weighted avg       0.52      0.52      0.52       570

Confusion Matrix:
[[ 96 151]
 [121 202]]


Moving Forward to new sentiment first using FinBERT

In [83]:
import json
import pandas as pd

file_path = "data/r_AusFinance_posts.jsonl"  # change if needed

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            post = json.loads(line)

            created = pd.to_datetime(post.get("created_utc"), unit="s", utc=True)

            title = post.get("title", "")
            selftext = post.get("selftext", "")

            rows.append({
                "datetime": created,
                "date": created.date(),
                "subreddit": post.get("subreddit"),
                "title": title,
                "selftext": selftext,
                "text": f"{title} {selftext}".strip(),
                "score": post.get("score"),
                "num_comments": post.get("num_comments"),
                "flair": post.get("link_flair_text"),
                "url": post.get("url"),
                "permalink": post.get("permalink")
            })

reddit_df = pd.DataFrame(rows)

reddit_df = reddit_df[
    reddit_df["text"].notna()
    & (reddit_df["text"].str.strip() != "")
    & (~reddit_df["text"].str.lower().isin(["[removed]", "[deleted]"]))
].reset_index(drop=True)

print("Shape:", reddit_df.shape)
print("Date range:", reddit_df["date"].min(), "→", reddit_df["date"].max())
print("Columns:", reddit_df.columns.tolist())
display(reddit_df.head())

Shape: (2216, 11)
Date range: 2022-01-01 → 2022-01-31
Columns: ['datetime', 'date', 'subreddit', 'title', 'selftext', 'text', 'score', 'num_comments', 'flair', 'url', 'permalink']


,datetime,date,subreddit,title,selftext,text,score,num_comments,flair,url,permalink
0,2022-01-01 00:06:10+00:00,2022-01-01,AusFinance,Understanding Equifax Credit Rating,"Hi all,\n\nSo I've decided to hit up the big 3...","Understanding Equifax Credit Rating Hi all,\n\...",0,19,Lifestyle,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rt6t6y/understanding_eq...
1,2022-01-01 00:10:45+00:00,2022-01-01,AusFinance,Good Budget,\n\nThanks to someone on the forum who recomme...,Good Budget \n\nThanks to someone on the forum...,0,0,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rt6wad/good_budget/
2,2022-01-01 00:20:07+00:00,2022-01-01,AusFinance,I’m a teen and I want to start investing in in...,I’m not that knowledgable about this field and...,I’m a teen and I want to start investing in in...,26,17,Investing,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rt72sb/im_a_teen_and_i_...
3,2022-01-01 00:38:15+00:00,2022-01-01,AusFinance,Is there something I can do on the side to mak...,[deleted],Is there something I can do on the side to mak...,31,51,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rt7ekh/is_there_somethi...
4,2022-01-01 00:55:14+00:00,2022-01-01,AusFinance,Assumed Inflation rate for Aged Care RAD?,"I am updating our long term plan (context, we ...",Assumed Inflation rate for Aged Care RAD? I am...,2,13,Business,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rt7pus/assumed_inflatio...


In [84]:
keywords_strict = [
    "asx",
    "stock",
    "stocks",
    "share",
    "shares",
    "trading",
    "market",
    "portfolio",
    "dividend",
    "earnings",
    "bull",
    "bear",
    "buy",
    "sell",
    "valuation",
    "cba",
    "commonwealth bank",
    "rba",
    "interest rate",
    "inflation"
]

In [85]:
exclude_keywords = [
    "property",
    "mortgage",
    "house",
    "rent",
    "salary",
    "job",
    "career",
    "budget",
    "saving",
    "superannuation",
    "tax return",
    "child",
    "family",
    "loan",
    "debt"
]

In [86]:
import re

include_pattern = re.compile("|".join(keywords_strict), re.IGNORECASE)
exclude_pattern = re.compile("|".join(exclude_keywords), re.IGNORECASE)

filtered_final = reddit_df[
    reddit_df["text"].str.contains(include_pattern, na=False) &
    ~reddit_df["text"].str.contains(exclude_pattern, na=False)
].reset_index(drop=True)

print("Final count:", len(filtered_final))
display(filtered_final.head(10))

Final count: 283


,datetime,date,subreddit,title,selftext,text,score,num_comments,flair,url,permalink
0,2022-01-01 04:41:25+00:00,2022-01-01,AusFinance,Record keeping - etf - vdhg - long term hold,As I am intending to keep vdhg for at least th...,Record keeping - etf - vdhg - long term hold A...,10,37,Investing,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtbnr5/record_keeping_e...
1,2022-01-01 09:50:02+00:00,2022-01-01,AusFinance,Should I still buy in IVV in 2022?,"Hi guys\n\nWhat’s you guys thought on IVV etf,...",Should I still buy in IVV in 2022? Hi guys\n\n...,2,26,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtg8xi/should_i_still_b...
2,2022-01-01 09:50:07+00:00,2022-01-01,AusFinance,Hey. Start investing with Spaceship Voyager to...,[removed],Hey. Start investing with Spaceship Voyager to...,1,1,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtg8z2/hey_start_invest...
3,2022-01-01 22:47:35+00:00,2022-01-01,AusFinance,"Are Raiz, Spaceship and CommSec Pocket worthwh...",Just curious if anyone uses these over the tra...,"Are Raiz, Spaceship and CommSec Pocket worthwh...",74,95,Superannuation,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtuq4j/are_raiz_spacesh...
4,2022-01-01 22:55:29+00:00,2022-01-01,AusFinance,Problem with businesses increasing profits,Someone over at r/Australia made this post:\n\...,Problem with businesses increasing profits Som...,10,85,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtuw07/problem_with_bus...
5,2022-01-02 01:10:27+00:00,2022-01-02,AusFinance,"Best brokers for Australia, European stock mar...",[removed],"Best brokers for Australia, European stock mar...",1,0,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtxp0b/best_brokers_for...
6,2022-01-02 02:16:49+00:00,2022-01-02,AusFinance,Do Australian politicians report their stock t...,Nancy Pelosi has been in the news a lot lately...,Do Australian politicians report their stock t...,35,11,Investing,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/rtz0e1/do_australian_po...
7,2022-01-02 05:03:59+00:00,2022-01-02,AusFinance,First home buyer - realistic costs?,[deleted],First home buyer - realistic costs? [deleted],13,46,Property,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/ru25qz/first_home_buyer...
8,2022-01-02 05:05:12+00:00,2022-01-02,AusFinance,Offset Account Balance - is it ok to consider ...,I personally think it can be considered as a b...,Offset Account Balance - is it ok to consider ...,2,9,Investing,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/ru26k7/offset_account_b...
9,2022-01-02 06:01:49+00:00,2022-01-02,AusFinance,Why no NDQ distribution this time?,The BetaShares NDQ 100 ETF isn't giving a dist...,Why no NDQ distribution this time? The BetaSha...,0,4,None,https://www.reddit.com/r/AusFinance/comments/r...,/r/AusFinance/comments/ru35st/why_no_ndq_distr...


In [87]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import pandas as pd

model_name = "ProsusAI/finbert"

tokenizer = BertTokenizer.from_pretrained(model_name)
finbert_model = BertForSequenceClassification.from_pretrained(model_name)

finbert_model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 51224.64it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [88]:
labels = ["positive", "negative", "neutral"]

def get_finbert_sentiment(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = finbert_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)[0].numpy()

    return {
        "positive_prob": probs[0],
        "negative_prob": probs[1],
        "neutral_prob": probs[2],
        "net_sentiment": probs[0] - probs[1]
    }

In [89]:
finbert_sample = filtered_final.head(50).copy()

sentiment_results = finbert_sample["text"].apply(get_finbert_sentiment)
sentiment_df = pd.DataFrame(list(sentiment_results))

finbert_sample = pd.concat(
    [finbert_sample.reset_index(drop=True), sentiment_df],
    axis=1
)

display(
    finbert_sample[
        ["date", "title", "positive_prob", "negative_prob", "neutral_prob", "net_sentiment"]
    ].head(10)
)

print("\nAverage sentiment:")
print(finbert_sample[["positive_prob", "negative_prob", "neutral_prob", "net_sentiment"]].mean())

,date,title,positive_prob,negative_prob,neutral_prob,net_sentiment
0,2022-01-01,Record keeping - etf - vdhg - long term hold,0.055320,0.025900,0.918779,0.029420
1,2022-01-01,Should I still buy in IVV in 2022?,0.156187,0.022098,0.821714,0.134089
2,2022-01-01,Hey. Start investing with Spaceship Voyager to...,0.045668,0.013699,0.940634,0.031969
3,2022-01-01,"Are Raiz, Spaceship and CommSec Pocket worthwh...",0.088122,0.013350,0.898528,0.074772
4,2022-01-01,Problem with businesses increasing profits,0.105355,0.039871,0.854774,0.065483
5,2022-01-02,"Best brokers for Australia, European stock mar...",0.041681,0.027861,0.930457,0.013820
6,2022-01-02,Do Australian politicians report their stock t...,0.033495,0.193288,0.773217,-0.159793
7,2022-01-02,First home buyer - realistic costs?,0.035716,0.094840,0.869444,-0.059124
8,2022-01-02,Offset Account Balance - is it ok to consider ...,0.054512,0.014260,0.931228,0.040253
9,2022-01-02,Why no NDQ distribution this time?,0.029522,0.090162,0.880315,-0.060640



Average sentiment:
positive_prob    0.054709
negative_prob    0.118677
neutral_prob     0.826614
net_sentiment   -0.063968
dtype: float32


In [90]:
sentiment_results = filtered_final["text"].apply(get_finbert_sentiment)

sentiment_df = pd.DataFrame(list(sentiment_results))

reddit_sentiment = pd.concat(
    [filtered_final.reset_index(drop=True), sentiment_df],
    axis=1
)

print("Shape:", reddit_sentiment.shape)

display(
    reddit_sentiment[
        ["date", "title", "positive_prob", "negative_prob", "neutral_prob", "net_sentiment"]
    ].head(10)
)

print("\nAverage sentiment:")
print(
    reddit_sentiment[
        ["positive_prob", "negative_prob", "neutral_prob", "net_sentiment"]
    ].mean()
)

Shape: (283, 15)


,date,title,positive_prob,negative_prob,neutral_prob,net_sentiment
0,2022-01-01,Record keeping - etf - vdhg - long term hold,0.055320,0.025900,0.918779,0.029420
1,2022-01-01,Should I still buy in IVV in 2022?,0.156187,0.022098,0.821714,0.134089
2,2022-01-01,Hey. Start investing with Spaceship Voyager to...,0.045668,0.013699,0.940634,0.031969
3,2022-01-01,"Are Raiz, Spaceship and CommSec Pocket worthwh...",0.088122,0.013350,0.898528,0.074772
4,2022-01-01,Problem with businesses increasing profits,0.105355,0.039871,0.854774,0.065483
5,2022-01-02,"Best brokers for Australia, European stock mar...",0.041681,0.027861,0.930457,0.013820
6,2022-01-02,Do Australian politicians report their stock t...,0.033495,0.193288,0.773217,-0.159793
7,2022-01-02,First home buyer - realistic costs?,0.035716,0.094840,0.869444,-0.059124
8,2022-01-02,Offset Account Balance - is it ok to consider ...,0.054512,0.014260,0.931228,0.040253
9,2022-01-02,Why no NDQ distribution this time?,0.029522,0.090162,0.880315,-0.060640



Average sentiment:
positive_prob    0.059676
negative_prob    0.150844
neutral_prob     0.789480
net_sentiment   -0.091168
dtype: float32


In [91]:
daily_sentiment = reddit_sentiment.groupby("date").agg({
    "positive_prob": "mean",
    "negative_prob": "mean",
    "neutral_prob": "mean",
    "net_sentiment": "mean",
    "text": "count"
}).rename(columns={"text": "post_count"}).reset_index()

print("Shape:", daily_sentiment.shape)
display(daily_sentiment.head())

print("\nDate range:")
print(daily_sentiment["date"].min(), "→", daily_sentiment["date"].max())

Shape: (31, 6)


,date,positive_prob,negative_prob,neutral_prob,net_sentiment,post_count
0,2022-01-01,0.090130,0.022984,0.886886,0.067147,5
1,2022-01-02,0.046135,0.151094,0.802772,-0.104959,7
2,2022-01-03,0.047733,0.052352,0.899915,-0.004619,4
3,2022-01-04,0.050044,0.055141,0.894815,-0.005096,6
4,2022-01-05,0.041177,0.096185,0.862639,-0.055008,10



Date range:
2022-01-01 → 2022-01-31


In [92]:
daily_sentiment["post_count"].describe()

count    31.000000
mean      9.129032
std       3.422883
min       3.000000
25%       6.500000
50%      10.000000
75%      12.000000
max      16.000000
Name: post_count, dtype: float64

In [93]:
# Ensure date formats match
CBA_Stock["date"] = pd.to_datetime(CBA_Stock["Date"]).dt.date
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"]).dt.date

# Merge
final_df = pd.merge(
    CBA_Stock,
    daily_sentiment,
    on="date",
    how="left"
)

print("Shape:", final_df.shape)
display(final_df.head())

Shape: (4098, 29)


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend,date,positive_prob,negative_prob,neutral_prob,net_sentiment,post_count
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,0.424591,0.679077,-0.024289,0,2010-01-18,NaN,NaN,NaN,NaN,NaN
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,-0.214539,0.004040,-0.001412,0,2010-01-19,NaN,NaN,NaN,NaN,NaN
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,-0.299812,-0.057893,0.000000,0,2010-01-20,NaN,NaN,NaN,NaN,NaN
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,-0.279167,-0.108163,-0.014321,0,2010-01-21,NaN,NaN,NaN,NaN,NaN
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,-0.431776,-0.453318,-0.010762,0,2010-01-22,NaN,NaN,NaN,NaN,NaN


In [94]:
print(final_df.isna().sum())

Date                   0
Close                  0
High                   0
Low                    0
Open                   0
Volume                 0
Ticker                 0
Return                 0
LogReturn              0
SMA_5                  0
SMA_10                 0
Return_std_5           0
Return_std_10          0
Return_lag1            0
Return_lag2            0
Return_lag3            0
Return_lag5            0
Momentum_5             0
Momentum_10            0
Close_SMA5_diff        0
Close_SMA10_diff       0
TargetReturn           0
TargetTrend            0
date                   0
positive_prob       4079
negative_prob       4079
neutral_prob        4079
net_sentiment       4079
post_count          4079
dtype: int64


In [95]:
jan2022_df = final_df[
    (final_df["date"] >= pd.to_datetime("2022-01-01").date()) &
    (final_df["date"] <= pd.to_datetime("2022-01-31").date())
].reset_index(drop=True)

print("Shape:", jan2022_df.shape)
display(jan2022_df.head())

Shape: (19, 29)


,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend,date,positive_prob,negative_prob,neutral_prob,net_sentiment,post_count
0,2022-01-04,87.162361,87.425946,86.133521,86.728716,2195465,CBA.AX,0.014951,0.014840,86.441321,...,0.721040,1.614687,0.006731,1,2022-01-04,0.050044,0.055141,0.894815,-0.005096,6.0
1,2022-01-05,87.749054,87.987138,87.255896,87.817078,2086996,CBA.AX,0.006731,0.006708,86.878366,...,0.870688,1.854465,-0.031298,0,2022-01-05,0.041177,0.096185,0.862639,-0.055008,10.0
2,2022-01-06,85.002655,87.689542,84.841099,87.408954,2603548,CBA.AX,-0.031298,-0.031799,86.550159,...,-1.547504,-0.957408,0.026808,1,2022-01-06,0.075042,0.074611,0.850347,0.000431,12.0
3,2022-01-07,87.281403,87.468466,86.635187,86.720214,2596039,CBA.AX,0.026808,0.026455,86.614781,...,0.666621,1.061153,0.006527,1,2022-01-07,0.044609,0.253702,0.701689,-0.209093,10.0
4,2022-01-10,87.851082,87.851082,86.796735,87.468458,1467407,CBA.AX,0.006527,0.006506,87.009311,...,0.841771,1.329833,-0.015099,0,2022-01-10,0.068347,0.063325,0.868328,0.005022,11.0


In [96]:
jan2022_df[[
    "positive_prob",
    "negative_prob",
    "neutral_prob",
    "net_sentiment",
    "post_count"
]] = jan2022_df[[
    "positive_prob",
    "negative_prob",
    "neutral_prob",
    "net_sentiment",
    "post_count"
]].fillna(0)

In [97]:
feature_columns_sentiment = [
    'Return',
    'Volume',
    'Return_lag1',
    'Return_lag2',
    'Return_lag3',
    'Return_lag5',
    'Momentum_5',
    'Momentum_10',
    'Return_std_5',
    'Return_std_10',
    'Close_SMA5_diff',
    'Close_SMA10_diff',

    # NEW FEATURES
    'net_sentiment',
    'post_count'
]

In [99]:
split_index = int(len(jan2022_df) * 0.7)

train_df = jan2022_df.iloc[:split_index]
test_df = jan2022_df.iloc[split_index:]

In [100]:
X_train = train_df[feature_columns_sentiment]
y_train = train_df["TargetTrend"]

X_test = test_df[feature_columns_sentiment]
y_test = test_df["TargetTrend"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, y_prob)

print("Out-of-sample AUC:", auc)

Out-of-sample AUC: 0.5555555555555556
